# 🔬 Autonomous Research Assistant — LangChain + LangGraph + ChatGroq (বাংলা)

এই নোটবুকে আমরা দেখব কীভাবে **Agentic AI** ব্যবহার করে একটি স্বয়ংক্রিয় গবেষণা সহায়ক তৈরি করা যায়।  
সিস্টেমটি একটি গবেষণার বিষয় নিয়ে, নিজে থেকে keyword বের করে, paper খুঁজে, বিশ্লেষণ করে এবং literature review তৈরি করে।

---

## এই নোটবুকে যা শিখব:

| ধাপ | বিষয় | বিবরণ |
|-----|-------|--------|
| ১ | **Keyword Extraction Agent** | LLM দিয়ে গবেষণার keyword বের করা |
| ২ | **Research Retrieval Agent** | arXiv থেকে paper খোঁজা |
| ৩ | **Paper Analysis Agent** | Abstract বিশ্লেষণ ও সারাংশ তৈরি |
| ৪ | **Literature Review Generator** | সম্পূর্ণ literature review document তৈরি |
| ৫ | **LangGraph Workflow** | সব agent একসাথে orchestrate করা |

---

### মূল ধারণা:
এই সিস্টেমে একটি মাত্র AI model নেই — বরং **একাধিক specialized agent** আছে, যারা একে অপরের সাথে কাজ করে।  
LangGraph ব্যবহার করে এই agents-দের মধ্যে workflow তৈরি করা হয়, আর ChatGroq দ্রুত inference এর জন্য ব্যবহার করা হয়।

## ১. প্রয়োজনীয় Library Install করা

প্রথমে আমাদের প্রজেক্টের জন্য দরকারি সব library install করতে হবে।

- `langchain` — AI chain ও agent তৈরির মূল framework
- `langgraph` — Agent-দের মধ্যে graph-based workflow তৈরি করে
- `langchain-groq` — Groq-এর fast inference API ব্যবহার করতে দেয়
- `arxiv` — arXiv থেকে research paper retrieve করার জন্য
- `python-dotenv` — API key সুরক্ষিতভাবে লোড করতে

In [4]:
!pip install langchain langgraph langchain-groq langchain-community arxiv python-dotenv pydantic -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## ২. Environment Setup ও API Key লোড করা

Groq API ব্যবহার করতে হলে একটি API key দরকার।  
[console.groq.com](https://console.groq.com) থেকে বিনামূল্যে key নেওয়া যায়।

`.env` ফাইলে রাখুন:
```
GROQ_API_KEY=your_key_here
```

অথবা নিচে সরাসরি বসিয়ে দিন (notebook শেষে মুছে দিন)।

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

# যদি .env ফাইল না থাকে, সরাসরি এখানে key বসান
# os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY:
    print("✅ Groq API Key সফলভাবে লোড হয়েছে।")
else:
    print("❌ API Key পাওয়া যায়নি। GROQ_API_KEY সেট করুন।")

✅ Groq API Key সফলভাবে লোড হয়েছে।


## ৩. সব প্রয়োজনীয় Import করা

এখানে আমরা প্রজেক্টের সব module এক জায়গায় import করব।

- `ChatGroq` — Groq-এর LLM model ব্যবহার করার জন্য
- `StateGraph` — LangGraph দিয়ে agent workflow তৈরি করতে
- `ChatPromptTemplate` — Prompt template তৈরি করতে
- `PydanticOutputParser` — LLM-এর output কে structured format-এ পেতে
- `arxiv` — Academic paper search করতে

In [6]:
import json
import arxiv
from typing import TypedDict, List, Optional, Annotated
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser

from langgraph.graph import StateGraph, END

print("✅ সব library সফলভাবে import হয়েছে।")

/Users/tappware/Desktop/Langchain-Github/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ সব library সফলভাবে import হয়েছে।


## ৪. ChatGroq LLM সেটআপ করা

### এই অংশে কী হচ্ছে?

`ChatGroq` হলো Groq-এর API ব্যবহার করার interface।  
Groq একটি অত্যন্ত দ্রুত inference engine — সাধারণ OpenAI-এর তুলনায় অনেক বেশি দ্রুত response দেয়।

আমরা `llama-3.3-70b-versatile` model ব্যবহার করব — এটি বড় ও শক্তিশালী, গবেষণা বিশ্লেষণের জন্য উপযুক্ত।

```
User Input
    ↓
ChatGroq (LLM)
    ↓
Structured Output
```

In [7]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    groq_api_key=GROQ_API_KEY
)

print("✅ ChatGroq LLM প্রস্তুত।")
print(f"   Model: llama-3.3-70b-versatile")
print(f"   Temperature: 0.3 (নির্ভুলতার জন্য কম রাখা হয়েছে)")

✅ ChatGroq LLM প্রস্তুত।
   Model: llama-3.3-70b-versatile
   Temperature: 0.3 (নির্ভুলতার জন্য কম রাখা হয়েছে)


## ৫. LangGraph State তৈরি করা

### এই অংশে কী হচ্ছে?

LangGraph-এ **State** হলো সেই তথ্যের পাত্র যা সব agent একে অপরের সাথে share করে।  
মনে করুন এটি একটি shared whiteboard — প্রতিটি agent এই whiteboard-এ তার কাজের ফলাফল লেখে, এবং পরের agent সেটি পড়ে।

আমাদের State-এ থাকবে:
- `research_topic` — ব্যবহারকারীর দেওয়া গবেষণার বিষয়
- `keywords` — Keyword Extraction Agent-এর output
- `papers` — Retrieved paper-গুলোর তালিকা
- `paper_summaries` — বিশ্লেষণ করা summaries
- `literature_review` — চূড়ান্ত literature review document

In [8]:
class ResearchState(TypedDict):
    research_topic: str
    keywords: List[str]
    papers: List[dict]
    paper_summaries: List[str]
    literature_review: str
    error: Optional[str]

print("✅ ResearchState তৈরি হয়েছে।")
print("   Fields:", list(ResearchState.__annotations__.keys()))

✅ ResearchState তৈরি হয়েছে।
   Fields: ['research_topic', 'keywords', 'papers', 'paper_summaries', 'literature_review', 'error']


## ৬. Pydantic Model দিয়ে Structured Output তৈরি করা

### এই অংশে কী হচ্ছে?

LLM সাধারণত plain text return করে। কিন্তু আমাদের দরকার **structured data** — যেমন JSON।

Pydantic `BaseModel` ব্যবহার করে আমরা LLM-এর output-কে একটি নির্দিষ্ট format-এ force করতে পারি।

```
LLM Output (plain text)
        ↓
PydanticOutputParser
        ↓
Structured Python Object
```

এতে করে পরের agent ঠিকঠাক data পায়, random text না।

In [9]:
class KeywordOutput(BaseModel):
    keywords: List[str] = Field(
        description="গবেষণার বিষয় থেকে বের করা গুরুত্বপূর্ণ keyword-গুলোর তালিকা"
    )
    search_queries: List[str] = Field(
        description="arXiv-এ search করার জন্য optimize করা query-গুলো"
    )

keyword_parser = PydanticOutputParser(pydantic_object=KeywordOutput)

print("✅ KeywordOutput model এবং parser তৈরি হয়েছে।")

✅ KeywordOutput model এবং parser তৈরি হয়েছে।


## ৭. Agent ১ — Keyword Extraction Agent

### এই অংশে কী হচ্ছে?

প্রথম agent-এর কাজ হলো ব্যবহারকারীর দেওয়া research topic থেকে গুরুত্বপূর্ণ keyword বের করা।

**কীভাবে কাজ করে:**

```
User Input: "AI in healthcare diagnosis"
        ↓
ChatPromptTemplate (system + human message)
        ↓
ChatGroq LLM
        ↓
PydanticOutputParser
        ↓
Output: { keywords: [...], search_queries: [...] }
```

`ChatPromptTemplate`-এ দুটি অংশ আছে:
- **system**: LLM-কে বলা হচ্ছে সে একজন research expert
- **human**: ব্যবহারকারীর topic এবং parser instructions

In [10]:
keyword_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert research keyword extractor.
Given a research topic, extract the most important academic keywords
and generate optimized search queries for arXiv.

{format_instructions}
"""),
    ("human", "Research topic: {topic}")
])

keyword_chain = keyword_prompt | llm | keyword_parser

def keyword_extraction_agent(state: ResearchState) -> ResearchState:
    print("🔍 Keyword Extraction Agent চলছে...")
    try:
        result = keyword_chain.invoke({
            "topic": state["research_topic"],
            "format_instructions": keyword_parser.get_format_instructions()
        })
        state["keywords"] = result.keywords
        print(f"   ✅ {len(result.keywords)} টি keyword বের হয়েছে: {result.keywords[:3]}...")
    except Exception as e:
        state["error"] = f"Keyword extraction error: {str(e)}"
        state["keywords"] = [state["research_topic"]]
        print(f"   ⚠️ Error: {e} — topic নিজেই keyword হিসেবে ব্যবহার হচ্ছে।")
    return state

print("✅ Keyword Extraction Agent তৈরি হয়েছে।")

✅ Keyword Extraction Agent তৈরি হয়েছে।


> **মূল কথা:** Keyword Extraction Agent একটি LLM chain ব্যবহার করে। `ChatPromptTemplate | ChatGroq | PydanticOutputParser` — এই তিনটি একসাথে জুড়ে দেওয়াই হলো LangChain-এর LCEL (LangChain Expression Language) pattern।

## ৮. Agent ২ — Research Retrieval Agent

### এই অংশে কী হচ্ছে?

দ্বিতীয় agent keyword পেয়ে **arXiv**-এ গিয়ে actual research paper খুঁজে আনে।

arXiv হলো বিশ্বের সবচেয়ে বড় free academic paper repository — physics, CS, math সহ অনেক বিষয়ে লক্ষ লক্ষ paper আছে।

**কীভাবে কাজ করে:**

```
Keywords: ["deep learning", "healthcare AI"]
        ↓
arxiv.Client().search(query)
        ↓
Top 3 papers per keyword
        ↓
Deduplicated list of papers
```

প্রতিটি paper থেকে আমরা রাখব: title, abstract, authors, year, এবং arXiv link।

In [11]:
def research_retrieval_agent(state: ResearchState) -> ResearchState:
    print("📚 Research Retrieval Agent চলছে...")
    keywords = state.get("keywords", [state["research_topic"]])
    all_papers = []
    seen_ids = set()

    client = arxiv.Client()

    for keyword in keywords[:3]:
        try:
            search = arxiv.Search(
                query=keyword,
                max_results=3,
                sort_by=arxiv.SortCriterion.Relevance
            )
            results = list(client.results(search))

            for paper in results:
                paper_id = paper.entry_id
                if paper_id not in seen_ids:
                    seen_ids.add(paper_id)
                    all_papers.append({
                        "title": paper.title,
                        "abstract": paper.summary[:500] + "...",
                        "authors": [a.name for a in paper.authors[:3]],
                        "year": paper.published.year,
                        "url": paper.entry_id,
                        "keyword": keyword
                    })
            print(f"   🔎 '{keyword}' — {len(results)} টি paper পাওয়া গেছে।")

        except Exception as e:
            print(f"   ⚠️ '{keyword}' search error: {e}")

    state["papers"] = all_papers
    print(f"   ✅ মোট {len(all_papers)} টি unique paper retrieve হয়েছে।")
    return state

print("✅ Research Retrieval Agent তৈরি হয়েছে।")

✅ Research Retrieval Agent তৈরি হয়েছে।


> **মূল কথা:** Research Retrieval Agent একটি external tool — arXiv API — ব্যবহার করে। এটিই Agentic AI-এর মূল বৈশিষ্ট্য: AI শুধু chat করে না, বাইরের দুনিয়া থেকে তথ্য এনে কাজ করে।

## ৯. Agent ৩ — Paper Analysis Agent

### এই অংশে কী হচ্ছে?

তৃতীয় agent প্রতিটি paper-এর title ও abstract পড়ে, বিশ্লেষণ করে এবং একটি বাংলা/English সারাংশ তৈরি করে।

এই agent প্রতিটি paper-এর জন্য আলাদাভাবে LLM-কে call করে:

```
Paper 1 → LLM Analysis → Summary 1
Paper 2 → LLM Analysis → Summary 2
Paper 3 → LLM Analysis → Summary 3
        ↓
List of summaries → পরের agent-এ পাঠানো হবে
```

Prompt-এ বলা হয়েছে summary যেন ৩টি অংশে থাকে:
- **Main Finding** — মূল আবিষ্কার কী?
- **Methodology** — কী পদ্ধতি ব্যবহার হয়েছে?
- **Contribution** — এটি গবেষণায় কী যোগ করেছে?

In [12]:
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert academic paper analyzer.
Analyze the given paper's title and abstract, then provide a concise structured summary.
Format your response as:
**Main Finding:** [what was discovered]
**Methodology:** [how it was done]
**Contribution:** [why it matters]
Keep each section to 1-2 sentences.
"""),
    ("human", """
Paper Title: {title}
Authors: {authors}
Year: {year}

Abstract: {abstract}

Provide a structured analysis.
""")
])

analysis_chain = analysis_prompt | llm | StrOutputParser()

def paper_analysis_agent(state: ResearchState) -> ResearchState:
    print("🧠 Paper Analysis Agent চলছে...")
    papers = state.get("papers", [])

    if not papers:
        state["paper_summaries"] = []
        print("   ⚠️ কোনো paper পাওয়া যায়নি।")
        return state

    summaries = []

    for i, paper in enumerate(papers[:6]):
        try:
            summary = analysis_chain.invoke({
                "title": paper["title"],
                "authors": ", ".join(paper["authors"]),
                "year": paper["year"],
                "abstract": paper["abstract"]
            })

            formatted_summary = f"""
### Paper {i+1}: {paper['title']}
**Authors:** {', '.join(paper['authors'])} ({paper['year']})
**URL:** {paper['url']}

{summary}
"""
            summaries.append(formatted_summary)
            print(f"   ✅ Paper {i+1} বিশ্লেষণ সম্পন্ন: {paper['title'][:50]}...")

        except Exception as e:
            print(f"   ⚠️ Paper {i+1} বিশ্লেষণে error: {e}")

    state["paper_summaries"] = summaries
    print(f"   ✅ মোট {len(summaries)} টি paper বিশ্লেষণ সম্পন্ন।")
    return state

print("✅ Paper Analysis Agent তৈরি হয়েছে।")

✅ Paper Analysis Agent তৈরি হয়েছে।


> **মূল কথা:** Paper Analysis Agent LLM-কে একটি structured analyst হিসেবে ব্যবহার করে। System prompt-এ তার role define করা হয়েছে, এবং human message-এ paper-এর data পাঠানো হয়। এই pattern-টি LangChain-এ সবচেয়ে বেশি ব্যবহৃত হয়।

## ১০. Agent ৪ — Literature Review Generator Agent

### এই অংশে কী হচ্ছে?

চতুর্থ এবং সর্বশেষ agent সব analyzed summaries নিয়ে একটি **সম্পূর্ণ Literature Review** document তৈরি করে।

এটিই পুরো pipeline-এর চূড়ান্ত output। Document-এ থাকবে:
- **Introduction** — গবেষণার পটভূমি
- **Existing Research** — পাওয়া papers-এর সারসংক্ষেপ
- **Key Findings** — মূল আবিষ্কারগুলো
- **Common Trends** — সাধারণ প্রবণতা
- **Research Gaps** — কোথায় আরও গবেষণা দরকার
- **Conclusion** — উপসংহার

```
Paper Summaries (6 টি)
        ↓
Combine into one big prompt
        ↓
ChatGroq LLM (generation)
        ↓
Complete Literature Review Document
```

In [13]:
review_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert academic writer specializing in literature reviews.
Write a comprehensive, well-structured literature review based on the provided paper summaries.

Structure your review with these sections:
1. **Introduction** - Background and importance of the topic
2. **Existing Research Overview** - Summary of the papers reviewed
3. **Key Findings** - Most important discoveries across papers
4. **Common Trends** - Patterns and trends identified
5. **Research Gaps** - What is missing or needs more study
6. **Conclusion** - Summary and future directions

Write in academic style. Be concise but comprehensive.
"""),
    ("human", """
Research Topic: {topic}
Keywords Analyzed: {keywords}

Paper Summaries:
{summaries}

Generate a complete literature review document.
""")
])

review_chain = review_prompt | llm | StrOutputParser()

def literature_review_generator(state: ResearchState) -> ResearchState:
    print("✍️ Literature Review Generator চলছে...")
    summaries = state.get("paper_summaries", [])

    if not summaries:
        state["literature_review"] = "কোনো paper পাওয়া যায়নি — literature review তৈরি করা সম্ভব হয়নি।"
        return state

    combined_summaries = "\n\n---\n\n".join(summaries)

    try:
        review = review_chain.invoke({
            "topic": state["research_topic"],
            "keywords": ", ".join(state.get("keywords", [])),
            "summaries": combined_summaries
        })

        header = f"""
# Literature Review: {state['research_topic']}

**Generated by:** Autonomous Research Assistant  
**Keywords Analyzed:** {', '.join(state.get('keywords', []))}  
**Papers Reviewed:** {len(summaries)}  

---

"""
        state["literature_review"] = header + review
        print("   ✅ Literature Review সফলভাবে তৈরি হয়েছে।")

    except Exception as e:
        state["literature_review"] = f"Error generating review: {str(e)}"
        print(f"   ❌ Error: {e}")

    return state

print("✅ Literature Review Generator তৈরি হয়েছে।")

✅ Literature Review Generator তৈরি হয়েছে।


> **মূল কথা:** Literature Review Generator সব agent-এর কাজের ফসল একসাথে নিয়ে চূড়ান্ত document তৈরি করে। এটিই পুরো multi-agent system-এর সবচেয়ে গুরুত্বপূর্ণ output।

## ১১. LangGraph দিয়ে Workflow তৈরি করা

### এই অংশে কী হচ্ছে?

এখন সব agent আলাদাভাবে তৈরি হয়ে গেছে। এবার **LangGraph** দিয়ে তাদের একটি workflow-এ যুক্ত করা হবে।

LangGraph একটি **directed graph** তৈরি করে — মানে কোন agent কার পরে চলবে তা নির্ধারণ করে:

```
START
  ↓
keyword_extraction  (Agent 1)
  ↓
research_retrieval  (Agent 2)
  ↓
paper_analysis      (Agent 3)
  ↓
literature_review   (Agent 4)
  ↓
END
```

**কেন LangGraph?**
- Agents-এর মধ্যে state automatically pass হয়
- Complex branching logic সহজে add করা যায়
- Error handling ও retry built-in support

In [14]:
from langgraph.graph import START

workflow = StateGraph(ResearchState)

workflow.add_node("keyword_extraction", keyword_extraction_agent)
workflow.add_node("research_retrieval", research_retrieval_agent)
workflow.add_node("paper_analysis", paper_analysis_agent)
workflow.add_node("literature_review", literature_review_generator)

workflow.add_edge(START, "keyword_extraction")
workflow.add_edge("keyword_extraction", "research_retrieval")
workflow.add_edge("research_retrieval", "paper_analysis")
workflow.add_edge("paper_analysis", "literature_review")
workflow.add_edge("literature_review", END)

research_app = workflow.compile()

print("✅ LangGraph Workflow সফলভাবে compile হয়েছে।")
print("   Nodes:", ["keyword_extraction", "research_retrieval", "paper_analysis", "literature_review"])
print("   Flow: START → Agent1 → Agent2 → Agent3 → Agent4 → END")

✅ LangGraph Workflow সফলভাবে compile হয়েছে।
   Nodes: ['keyword_extraction', 'research_retrieval', 'paper_analysis', 'literature_review']
   Flow: START → Agent1 → Agent2 → Agent3 → Agent4 → END


> **মূল কথা:** `StateGraph` হলো LangGraph-এর মূল class। `add_node()` দিয়ে agent যোগ করা হয়, `add_edge()` দিয়ে তাদের মধ্যে সংযোগ তৈরি হয়, এবং `compile()` দিয়ে পুরো workflow execute করার জন্য প্রস্তুত হয়।

## ১২. সিস্টেম চালানো — Research Topic দিয়ে পরীক্ষা করা

### এই অংশে কী হচ্ছে?

এখন পুরো pipeline পরীক্ষা করার সময়! একটি research topic দিলে সিস্টেম:

1. Keyword বের করবে
2. arXiv থেকে paper আনবে  
3. প্রতিটি paper বিশ্লেষণ করবে
4. সম্পূর্ণ literature review তৈরি করবে

আপনি `research_topic` পরিবর্তন করে যেকোনো বিষয়ে চেষ্টা করতে পারেন।

In [15]:
research_topic = "transformer models in computer vision"

print("=" * 60)
print(f"🚀 গবেষণার বিষয়: {research_topic}")
print("=" * 60)
print()

initial_state: ResearchState = {
    "research_topic": research_topic,
    "keywords": [],
    "papers": [],
    "paper_summaries": [],
    "literature_review": "",
    "error": None
}

final_state = research_app.invoke(initial_state)

print()
print("=" * 60)
print("✅ সম্পূর্ণ Research Workflow সফলভাবে সম্পন্ন হয়েছে!")
print("=" * 60)

🚀 গবেষণার বিষয়: transformer models in computer vision

🔍 Keyword Extraction Agent চলছে...
   ✅ 10 টি keyword বের হয়েছে: ['Transformer Models', 'Computer Vision', 'Deep Learning']...
📚 Research Retrieval Agent চলছে...
   🔎 'Transformer Models' — 3 টি paper পাওয়া গেছে।
   🔎 'Computer Vision' — 3 টি paper পাওয়া গেছে।
   🔎 'Deep Learning' — 3 টি paper পাওয়া গেছে।
   ✅ মোট 9 টি unique paper retrieve হয়েছে।
🧠 Paper Analysis Agent চলছে...
   ✅ Paper 1 বিশ্লেষণ সম্পন্ন: PyramidTNT: Improved Transformer-in-Transformer Ba...
   ✅ Paper 2 বিশ্লেষণ সম্পন্ন: Glance-and-Gaze Vision Transformer...
   ✅ Paper 3 বিশ্লেষণ সম্পন্ন: Learning to Cluster Faces via Transformer...
   ✅ Paper 4 বিশ্লেষণ সম্পন্ন: WiCV 2019: The Sixth Women In Computer Vision Work...
   ✅ Paper 5 বিশ্লেষণ সম্পন্ন: Spatial Monitoring and Insect Behavioural Analysis...
   ✅ Paper 6 বিশ্লেষণ সম্পন্ন: Global Adaptive Filtering Layer for Computer Visio...
   ✅ মোট 6 টি paper বিশ্লেষণ সম্পন্ন।
✍️ Literature Review Generator চলছে

## ১৩. ফলাফল দেখা — Keywords ও Papers

এখন দেখা যাক প্রতিটি agent কী output দিয়েছে।

In [16]:
print("📌 বের হওয়া Keywords:")
for i, kw in enumerate(final_state["keywords"], 1):
    print(f"   {i}. {kw}")

print()
print(f"📄 মোট Papers পাওয়া গেছে: {len(final_state['papers'])} টি")
print()
print("প্রথম ৩টি Paper:")
for paper in final_state["papers"][:3]:
    print(f"   🔹 {paper['title'][:70]}... ({paper['year']})")

📌 বের হওয়া Keywords:
   1. Transformer Models
   2. Computer Vision
   3. Deep Learning
   4. Image Processing
   5. Vision Transformers
   6. Convolutional Neural Networks
   7. Attention Mechanism
   8. Object Detection
   9. Image Classification
   10. Segmentation

📄 মোট Papers পাওয়া গেছে: 9 টি

প্রথম ৩টি Paper:
   🔹 PyramidTNT: Improved Transformer-in-Transformer Baselines with Pyramid... (2022)
   🔹 Glance-and-Gaze Vision Transformer... (2021)
   🔹 Learning to Cluster Faces via Transformer... (2021)


## ১৪. চূড়ান্ত Literature Review দেখা

এটিই পুরো সিস্টেমের মূল output — একটি সম্পূর্ণ academic literature review document।

In [17]:
from IPython.display import Markdown, display

print("📋 সম্পূর্ণ Literature Review:")
print("=" * 60)
display(Markdown(final_state["literature_review"]))

📋 সম্পূর্ণ Literature Review:



# Literature Review: transformer models in computer vision

**Generated by:** Autonomous Research Assistant  
**Keywords Analyzed:** Transformer Models, Computer Vision, Deep Learning, Image Processing, Vision Transformers, Convolutional Neural Networks, Attention Mechanism, Object Detection, Image Classification, Segmentation  
**Papers Reviewed:** 6  

---

**Introduction**

The field of computer vision has witnessed significant advancements in recent years, driven by the development of deep learning techniques, particularly transformer models. Transformer models, initially designed for natural language processing tasks, have been successfully applied to computer vision tasks, such as image classification, object detection, and segmentation. The integration of transformer models in computer vision has led to improved performance and efficiency in various applications. This literature review aims to provide an overview of the current state of research on transformer models in computer vision, highlighting key findings, trends, and gaps in the existing literature.

**Existing Research Overview**

A comprehensive review of six papers on transformer models in computer vision reveals a diverse range of topics and methodologies. Paper 1, "PyramidTNT: Improved Transformer-in-Transformer Baselines with Pyramid Architecture," proposes a modified transformer-in-transformer architecture, PyramidTNT, which achieves improved performance by establishing hierarchical representations (Han et al., 2022). Paper 2, "Glance-and-Gaze Vision Transformer," introduces an efficient vision transformer that addresses the computational limitations of existing models, highlighting the trade-off between performance and computational costs (Yu et al., 2021). Paper 3, "Learning to Cluster Faces via Transformer," presents a face transformer model for supervised face clustering, which effectively handles variations in face poses, occlusions, and image quality (Ye et al., 2021).

In addition to these technical papers, the Women in Computer Vision Workshop (WiCV 2019) aimed to promote diversity and inclusion in the field, highlighting the underrepresentation of female researchers in academia and industry (Amerini et al., 2019). Paper 5, "Spatial Monitoring and Insect Behavioural Analysis Using Computer Vision for Precision Pollination," explores the application of computer vision in precision pollination, providing a detailed understanding of insect distributions and behavior (Ratnayake et al., 2022). Finally, Paper 6, "Global Adaptive Filtering Layer for Computer Vision," proposes a universal adaptive neural layer that can improve the performance of base neural networks in computer vision tasks by selecting the best frequencies for each image (Shipitsin et al., 2020).

**Key Findings**

The reviewed papers highlight several key findings in the application of transformer models in computer vision. Firstly, the introduction of pyramid architectures and convolutional stems can significantly improve the performance of transformer models (Han et al., 2022). Secondly, efficient vision transformers can be designed to address computational limitations, enabling real-world applications with limited resources (Yu et al., 2021). Thirdly, transformer models can be effectively applied to face clustering tasks, handling variations in face poses, occlusions, and image quality (Ye et al., 2021). Finally, computer vision techniques can be used in various applications, such as precision pollination, to improve crop production and food security (Ratnayake et al., 2022).

**Common Trends**

Several trends emerge from the reviewed papers. Firstly, there is a growing interest in developing efficient and effective transformer models for computer vision tasks, driven by the need for improved performance and reduced computational costs. Secondly, the application of transformer models in various computer vision tasks, such as image classification, object detection, and segmentation, is becoming increasingly popular. Thirdly, the use of adaptive neural layers and frequency domain processing is being explored to improve the performance of base neural networks in computer vision tasks (Shipitsin et al., 2020). Finally, there is a recognition of the importance of diversity and inclusion in the field of computer vision, with initiatives such as the Women in Computer Vision Workshop aiming to promote female participation and visibility (Amerini et al., 2019).

**Research Gaps**

Despite the significant advancements in transformer models for computer vision, several research gaps remain. Firstly, there is a need for further research on efficient and effective transformer models that can be applied to real-world applications with limited resources. Secondly, the development of transformer models for specific computer vision tasks, such as face clustering and precision pollination, requires further exploration. Thirdly, the integration of transformer models with other deep learning techniques, such as convolutional neural networks, is an area that requires further investigation. Finally, there is a need for more diverse and inclusive research communities, with initiatives such as the Women in Computer Vision Workshop playing a crucial role in promoting diversity and inclusion in the field.

**Conclusion**

In conclusion, the literature review highlights the significant advancements in transformer models for computer vision, with a growing interest in developing efficient and effective models for various applications. The key findings, trends, and research gaps identified in this review provide a foundation for future research in this area. As the field of computer vision continues to evolve, it is essential to address the research gaps and promote diversity and inclusion in the research community. Future directions for research include the development of more efficient and effective transformer models, the exploration of new applications, and the integration of transformer models with other deep learning techniques. Ultimately, the advancement of transformer models in computer vision has the potential to drive innovation and improvement in various applications, from image classification and object detection to precision pollination and beyond.

## ১৫. Literature Review ফাইলে সংরক্ষণ করা

### এই অংশে কী হচ্ছে?

তৈরি হওয়া literature review একটি Markdown ফাইলে save করা হচ্ছে।  
পরে এটি PDF বা DOCX-এ convert করা যাবে।

ফাইলের নামে topic এবং date যোগ করা হয়েছে — যাতে একাধিক research সহজে আলাদা করা যায়।

In [18]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_topic = research_topic.replace(" ", "_")[:30]
filename = f"literature_review_{safe_topic}_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(final_state["literature_review"])

print(f"✅ Literature Review সংরক্ষিত হয়েছে: {filename}")
print(f"   ফাইলের আকার: {len(final_state['literature_review'])} characters")

✅ Literature Review সংরক্ষিত হয়েছে: literature_review_transformer_models_in_computer_20260511_232810.md
   ফাইলের আকার: 6296 characters


## ১৬. নিজের Topic দিয়ে পরীক্ষা করুন

এখন আপনি চাইলে যেকোনো বিষয়ে research করতে পারেন।  
নিচের `custom_topic` পরিবর্তন করুন এবং cell run করুন।

কিছু উদাহরণ:
- `"large language models fine-tuning"`
- `"reinforcement learning robotics"`
- `"federated learning privacy"`
- `"natural language processing Bengali"`

In [21]:
custom_topic = "i have a research idea about some real life problems. idea like i want to research some thinking new on polysemous vs monosemous analysis . how llm models are detect or identify in word embeddings space"

print(f"🚀 নতুন Topic: {custom_topic}")
print("পুরো pipeline শুরু হচ্ছে...\n")

custom_state: ResearchState = {
    "research_topic": custom_topic,
    "keywords": [],
    "papers": [],
    "paper_summaries": [],
    "literature_review": "",
    "error": None
}

custom_result = research_app.invoke(custom_state)

print("\n" + "=" * 60)
display(Markdown(custom_result["literature_review"]))

🚀 নতুন Topic: i have a research idea about some real life problems. idea like i want to research some thinking new on polysemous vs monosemous analysis . how llm models are detect or identify in word embeddings space
পুরো pipeline শুরু হচ্ছে...

🔍 Keyword Extraction Agent চলছে...
   ✅ 8 টি keyword বের হয়েছে: ['Polysemous words', 'Monosemous words', 'Word embeddings']...
📚 Research Retrieval Agent চলছে...
   🔎 'Polysemous words' — 3 টি paper পাওয়া গেছে।
   🔎 'Monosemous words' — 3 টি paper পাওয়া গেছে।
   🔎 'Word embeddings' — 3 টি paper পাওয়া গেছে।
   ✅ মোট 9 টি unique paper retrieve হয়েছে।
🧠 Paper Analysis Agent চলছে...
   ✅ Paper 1 বিশ্লেষণ সম্পন্ন: A Simple Approach to Learn Polysemous Word Embeddi...
   ✅ Paper 2 বিশ্লেষণ সম্পন্ন: Rethinking Evaluation of Sparse Autoencoders throu...
   ✅ Paper 3 বিশ্লেষণ সম্পন্ন: Schrödinger's Bat: Diffusion Models Sometimes Gene...
   ✅ Paper 4 বিশ্লেষণ সম্পন্ন: On arch factorization and subword universality for...
   ✅ Paper 5 বিশ্লেষণ সম্পন


# Literature Review: i have a research idea about some real life problems. idea like i want to research some thinking new on polysemous vs monosemous analysis . how llm models are detect or identify in word embeddings space

**Generated by:** Autonomous Research Assistant  
**Keywords Analyzed:** Polysemous words, Monosemous words, Word embeddings, LLM models, Natural Language Processing, Semantic analysis, Word sense induction, Distributional semantics  
**Papers Reviewed:** 6  

---

**Introduction**

The study of polysemous and monosemous words has been a longstanding topic of interest in the field of Natural Language Processing (NLP). Polysemous words, which have multiple related meanings, and monosemous words, which have a single meaning, pose significant challenges for language models and word embeddings. The ability to accurately detect and identify these words is crucial for various NLP applications, including text classification, sentiment analysis, and machine translation. Recent advances in Large Language Models (LLMs) and word embeddings have led to significant improvements in NLP tasks, but the analysis of polysemous and monosemous words remains an open research area. This literature review aims to provide an overview of the existing research on polysemous and monosemous word analysis, with a focus on word embeddings and LLMs.

**Existing Research Overview**

The reviewed papers provide a comprehensive overview of the current state of research on polysemous and monosemous word analysis. Paper 1 proposes a simple approach to learn polysemous word embeddings, which can effectively disambiguate words with multiple meanings (Sun et al., 2017). Paper 2 challenges traditional evaluation methods for sparse autoencoders (SAEs), highlighting their limitations in assessing the semantic representational power of SAEs, particularly for polysemous words (Minegishi et al., 2025). Paper 3 investigates the phenomenon of text-to-image diffusion models generating images that contain multiple meanings of a polysemous word, demonstrating the limitations and quirks of these models (White & Cotterell, 2022). Papers 4, 5, and 6 focus on more theoretical aspects of word analysis, including the development of algorithms for computing subword universality indexes (Schnoebelen & Veron, 2023), the study of subword complexity in infinite partial words (Blanchet-Sadri et al., 2011), and the classification of trapezoidal words (Fici, 2011).

**Key Findings**

The reviewed papers have several key findings that contribute to our understanding of polysemous and monosemous word analysis. First, the use of word embeddings and LLMs can effectively disambiguate polysemous words, but more research is needed to improve the accuracy and efficiency of these methods. Second, traditional evaluation methods for SAEs may not be sufficient to assess their semantic representational power, particularly for polysemous words. Third, text-to-image diffusion models can generate images that contain multiple meanings of a polysemous word, highlighting the need for more research on the limitations and quirks of these models. Finally, the study of subword complexity and the classification of trapezoidal words can provide valuable insights into the structural properties of words and their applications in NLP.

**Common Trends**

Several common trends emerge from the reviewed papers. First, there is a growing interest in the use of word embeddings and LLMs for polysemous and monosemous word analysis. Second, the development of more effective evaluation methods for SAEs and other language models is a pressing need. Third, the study of polysemous words is becoming increasingly important, as these words pose significant challenges for language models and NLP applications. Finally, the intersection of NLP and other fields, such as computer science and linguistics, is leading to new insights and approaches to word analysis.

**Research Gaps**

Despite the significant progress made in polysemous and monosemous word analysis, several research gaps remain. First, more research is needed to improve the accuracy and efficiency of word embeddings and LLMs for polysemous word disambiguation. Second, the development of more effective evaluation methods for SAEs and other language models is a pressing need. Third, the study of polysemous words in different languages and contexts is an area that requires more attention. Finally, the application of word analysis techniques to real-world NLP problems, such as text classification and sentiment analysis, is an area that requires more research.

**Conclusion**

In conclusion, the reviewed papers provide a comprehensive overview of the current state of research on polysemous and monosemous word analysis. The use of word embeddings and LLMs, the development of more effective evaluation methods for SAEs, and the study of polysemous words are all areas that require more research. The intersection of NLP and other fields, such as computer science and linguistics, is leading to new insights and approaches to word analysis. Future research should focus on addressing the research gaps identified in this review, including the improvement of word embeddings and LLMs, the development of more effective evaluation methods, and the application of word analysis techniques to real-world NLP problems. Ultimately, the goal of polysemous and monosemous word analysis is to improve the accuracy and efficiency of NLP applications, and to provide a deeper understanding of the structure and properties of language.

---

## ১৭. সারাংশ — আমরা কী শিখলাম?

এই নোটবুকে আমরা একটি সম্পূর্ণ **Autonomous Research Assistant** তৈরি করলাম।

| Agent | কাজ | Technology |
|-------|-----|------------|
| Keyword Extraction Agent | Research topic থেকে keyword বের করা | ChatGroq + PydanticOutputParser |
| Research Retrieval Agent | arXiv থেকে paper আনা | arxiv API |
| Paper Analysis Agent | Abstract বিশ্লেষণ করা | ChatGroq + ChatPromptTemplate |
| Literature Review Generator | সম্পূর্ণ document তৈরি | ChatGroq + StrOutputParser |
| **Orchestration** | সব agent একসাথে চালানো | **LangGraph StateGraph** |

---

### 🔮 ভবিষ্যতে যা যোগ করা যাবে:

- **Reflection Agent** — Summary-এর মান যাচাই করবে
- **Citation Generator** — APA/MLA citation তৈরি করবে
- **Semantic Scholar + PubMed** — আরও বেশি database থেকে paper আনবে
- **Vector Database** — ChromaDB/Pinecone দিয়ে paper store করে RAG করবে
- **Human-in-the-loop** — প্রতিটি ধাপে মানুষের approval নেবে
- **Parallel Agents** — একসাথে একাধিক keyword search করবে

---

> **মূল কথা:** এই প্রজেক্ট Agentic AI-এর শক্তি দেখায় — একটি research topic দিলে পুরো workflow স্বয়ংক্রিয়ভাবে চলে এবং একটি professional document তৈরি হয়। LangGraph দিয়ে এই agents-দের orchestrate করা LangChain-এর LCEL pattern-এর একটি উন্নত রূপ।